# Thesis Data Profiling — Key Findings
**Project:** Moderation Dynamics in Transition — TikTok vs. X, Germany, 2025 Federal Snap Election  
**Author:** Mohsen Zahedi (Mat.-Nr. 6841065)  
**Created:** 2026-06-08  
**Source notebooks:** `temp/profiling_report.ipynb` + `temp/profiling_findings.ipynb`

---

This notebook consolidates **only the thesis-relevant findings** from the raw data profiling phase.  
It is structured around the six dimensions that directly bear on RQ1 and RQ2:  
decision ground, automation, content category, decision type, territorial scope, and source type.  
Data quality issues and cleaning decisions relevant to these dimensions are included.  
Visualisation code and low-level schema/cardinality detail are omitted — see source notebooks if needed.

> **Coverage:**  
> - TikTok: sample of 20 files → **15,439,990 rows** (Jan–Nov 2025) out of 1,197 files / 51.5 GB  
> - X: full dataset → **670,093 rows** (43 MB / 291 files)

---
## 0  Setup

In [3]:
import polars as pl
from pathlib import Path

pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(150)
pl.Config.set_tbl_width_chars(300)

OUT = Path(r'c:\Users\MoZa\OneDrive - Universität Paderborn\0_UPB\BA\Repo\Bachelor-Arbeit\temp\profiling_output')
COLORS = {'tiktok': '#EE1D52', 'x': '#1DA1F2'}

tt_null = pl.read_csv(OUT / 'tiktok_04_null_empty.csv')
x_null  = pl.read_csv(OUT / 'x_04_null_empty.csv')
tt_card = pl.read_csv(OUT / 'tiktok_05_cardinality.csv')
x_card  = pl.read_csv(OUT / 'x_05_cardinality.csv')
tt_topn = pl.read_csv(OUT / 'tiktok_06_topn_values.csv')
x_topn  = pl.read_csv(OUT / 'x_06_topn_values.csv')

def topn(df, col): return df.filter(pl.col('column') == col).sort('rank')
print('Ready.')

Ready.


---
## 1  Dataset Overview

Both platforms submit under the same DSA Statement of Reasons (SoR) schema — **37 columns**, identical structure.  
Despite this, the datasets differ dramatically in volume, file size, and how each field is actually used.

In [4]:
overview = pl.DataFrame([
    {'platform': 'TikTok', 'files_on_disk': 1197, 'size_gb': 51.5,
     'rows_profiled': 15_439_990, 'coverage': 'sample 20 files', 'columns': 37},
    {'platform': 'X',      'files_on_disk': 291,  'size_gb': 0.04,
     'rows_profiled': 670_093,    'coverage': 'full dataset',    'columns': 37},
])
print(overview)

shape: (2, 6)
┌──────────┬───────────────┬─────────┬───────────────┬─────────────────┬─────────┐
│ platform ┆ files_on_disk ┆ size_gb ┆ rows_profiled ┆ coverage        ┆ columns │
│ ---      ┆ ---           ┆ ---     ┆ ---           ┆ ---             ┆ ---     │
│ str      ┆ i64           ┆ f64     ┆ i64           ┆ str             ┆ i64     │
╞══════════╪═══════════════╪═════════╪═══════════════╪═════════════════╪═════════╡
│ TikTok   ┆ 1197          ┆ 51.5    ┆ 15439990      ┆ sample 20 files ┆ 37      │
│ X        ┆ 291           ┆ 0.04    ┆ 670093        ┆ full dataset    ┆ 37      │
└──────────┴───────────────┴─────────┴───────────────┴─────────────────┴─────────┘


**Thesis note.**  
The volume asymmetry (~23× more TikTok rows) is partly a reporting scope artefact, not purely a reflection of enforcement intensity.  
TikTok reports routine policy enforcement at scale (proactive, automated); X reports only legally-mandated illegal-content decisions, each scoped to a single country.  
This distinction is fundamental for RQ1 and must be foregrounded in §3.1 (Data Source & Scope).

---
## 2  Decision Ground — The Core Structural Difference (→ RQ1)

`decision_ground` encodes whether the moderation action was taken for *illegal content* (EU legal basis)  
or *incompatible content* (platform policy violation). This is the most analytically consequential column.

In [5]:
print('--- TikTok decision_ground ---')
print(topn(tt_topn, 'decision_ground').select(['value', 'count', 'pct']))

print('\n--- X decision_ground ---')
print(topn(x_topn, 'decision_ground').select(['value', 'count', 'pct']))

--- TikTok decision_ground ---
shape: (2, 3)
┌──────────────────────────────────────┬──────────┬───────┐
│ value                                ┆ count    ┆ pct   │
│ ---                                  ┆ ---      ┆ ---   │
│ str                                  ┆ i64      ┆ f64   │
╞══════════════════════════════════════╪══════════╪═══════╡
│ DECISION_GROUND_INCOMPATIBLE_CONTENT ┆ 15437962 ┆ 99.99 │
│ DECISION_GROUND_ILLEGAL_CONTENT      ┆ 2028     ┆ 0.01  │
└──────────────────────────────────────┴──────────┴───────┘

--- X decision_ground ---
shape: (1, 3)
┌─────────────────────────────────┬────────┬───────┐
│ value                           ┆ count  ┆ pct   │
│ ---                             ┆ ---    ┆ ---   │
│ str                             ┆ i64    ┆ f64   │
╞═════════════════════════════════╪════════╪═══════╡
│ DECISION_GROUND_ILLEGAL_CONTENT ┆ 670093 ┆ 100.0 │
└─────────────────────────────────┴────────┴───────┘


**Finding.**  
- **TikTok:** 99.99% `DECISION_GROUND_INCOMPATIBLE_CONTENT` — own platform policy violations. Only 0.01% cite an EU legal basis.  
- **X:** 100% `DECISION_GROUND_ILLEGAL_CONTENT` — every single entry cites an EU legal ground. `decision_ground` is a constant column for X (only 1 unique value).

**Implication for thesis.**  
This is not a data quality issue — it is a fundamental reporting scope difference.  
X submits only legally-mandated notices; TikTok submits the vast majority of its routine policy enforcement.  
Direct cross-platform comparison of raw counts or category distributions is therefore methodologically invalid without this caveat.  
Must be addressed prominently in §3.1 and §5.3 (Limitations).

---
## 3  Automation — Detection and Decision (→ RQ1, H1)

`automated_detection`: was the content flagged by AI/automation?  
`automated_decision`: was the enforcement decision made by automation?

In [6]:
for col in ['automated_detection', 'automated_decision']:
    print(f'\n--- TikTok {col} ---')
    print(topn(tt_topn, col).select(['value', 'count', 'pct']))
    print(f'\n--- X {col} ---')
    print(topn(x_topn, col).select(['value', 'count', 'pct']))


--- TikTok automated_detection ---
shape: (2, 3)
┌───────┬──────────┬───────┐
│ value ┆ count    ┆ pct   │
│ ---   ┆ ---      ┆ ---   │
│ str   ┆ i64      ┆ f64   │
╞═══════╪══════════╪═══════╡
│ Yes   ┆ 15284158 ┆ 98.99 │
│ No    ┆ 155832   ┆ 1.01  │
└───────┴──────────┴───────┘

--- X automated_detection ---
shape: (1, 3)
┌───────┬────────┬───────┐
│ value ┆ count  ┆ pct   │
│ ---   ┆ ---    ┆ ---   │
│ str   ┆ i64    ┆ f64   │
╞═══════╪════════╪═══════╡
│ No    ┆ 670093 ┆ 100.0 │
└───────┴────────┴───────┘

--- TikTok automated_decision ---
shape: (3, 3)
┌──────────────────────────────────┬──────────┬───────┐
│ value                            ┆ count    ┆ pct   │
│ ---                              ┆ ---      ┆ ---   │
│ str                              ┆ i64      ┆ f64   │
╞══════════════════════════════════╪══════════╪═══════╡
│ AUTOMATED_DECISION_FULLY         ┆ 14803399 ┆ 95.88 │
│ AUTOMATED_DECISION_NOT_AUTOMATED ┆ 636582   ┆ 4.12  │
│ AUTOMATED_DECISION_PARTIALLY     ┆ 9     

**Finding.**  
- **TikTok:** 98.99% of content was flagged by automation; 95.88% of decisions were fully automated. The enforcement pipeline is almost entirely machine-driven.  
- **X:** 100% of detections are `No` (not automated); 98.55% of decisions are not automated. Every X entry was actioned by a human agent.  
- `automated_detection` is a constant column for X (only 1 unique value: `No`).

**Implication for thesis.**  
This near-total divergence in automation rates directly supports H1 but cannot be interpreted naively.  
It reflects the reporting scope difference established in Section 2 — X only reports illegal-content decisions, which require human judgment under the DSA; TikTok reports mass automated policy enforcement.  
A direct numerical comparison of automation rates across platforms risks conflating operational practice with reporting scope artefact.  
The analysis in §4 must isolate within-platform temporal variation (pre/during/post election) rather than treating the cross-platform gap as a simple empirical finding.

---
## 4  Content Category Distribution (→ RQ1, RQ2)

The `category` field classifies the type of harmful content. Distributions reveal each platform's enforcement focus  
and are the primary variable for detecting election-period shifts (RQ2).

In [7]:
print('--- TikTok top categories ---')
print(topn(tt_topn, 'category').select(['rank', 'value', 'count', 'pct']))

print('\n--- X top categories ---')
print(topn(x_topn, 'category').select(['rank', 'value', 'count', 'pct']))

--- TikTok top categories ---
shape: (10, 4)
┌──────┬─────────────────────────────────────────────────────────────────────┬─────────┬───────┐
│ rank ┆ value                                                               ┆ count   ┆ pct   │
│ ---  ┆ ---                                                                 ┆ ---     ┆ ---   │
│ i64  ┆ str                                                                 ┆ i64     ┆ f64   │
╞══════╪═════════════════════════════════════════════════════════════════════╪═════════╪═══════╡
│ 1    ┆ STATEMENT_CATEGORY_ILLEGAL_OR_HARMFUL_SPEECH                        ┆ 6780854 ┆ 43.97 │
│ 2    ┆ STATEMENT_CATEGORY_SCOPE_OF_PLATFORM_SERVICE                        ┆ 5718881 ┆ 37.09 │
│ 3    ┆ STATEMENT_CATEGORY_OTHER_VIOLATION_TC                               ┆ 1152790 ┆ 7.48  │
│ 4    ┆ STATEMENT_CATEGORY_NEGATIVE_EFFECTS_ON_CIVIC_DISCOURSE_OR_ELECTIONS ┆ 660443  ┆ 4.28  │
│ 5    ┆ STATEMENT_CATEGORY_VIOLENCE                                         ┆ 520

**Finding.**  
- **TikTok top categories:** Illegal/Harmful Speech (44%), Scope of Platform Service (37%), Other ToC violation (7.5%), Civic Discourse/Elections (4.3%), Violence (3.4%).  
- **X top categories:** Pornography/Sexualized Content (21%), Scope of Service (18%), Protection of Minors (15%), Scams & Fraud (13%), Violence (10%).

TikTok's dominant category is speech-related; X's is sexualized content and child protection.  
These are structurally different distributions — not just quantitatively but qualitatively — reflecting each platform's enforcement focus and the reporting scope difference from Section 2.

**Thesis note.**  
The `Civic Discourse/Elections` category (4.3% on TikTok) is analytically central to RQ2.  
Temporal disaggregation of this category around the February 2025 snap election period is the core of the event-dynamics analysis.  
Note also that the v1→v2 schema break in July 2025 affects category field names — see taxonomy mapping (`taxonomy_mapping_v1_v2.csv`) before any longitudinal analysis.

---
## 5  Decision Type — What Action Was Taken (→ RQ1)

`decision_visibility`, `decision_provision`, and `decision_account` jointly describe the enforcement action.

In [8]:
for col in ['decision_visibility', 'decision_account']:
    print(f'\n--- TikTok {col} ---')
    print(topn(tt_topn, col).select(['value', 'count', 'pct']))
    print(f'\n--- X {col} ---')
    print(topn(x_topn, col).select(['value', 'count', 'pct']))


--- TikTok decision_visibility ---
shape: (10, 3)
┌──────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬───────┐
│ value                                                                                                            ┆ count   ┆ pct   │
│ ---                                                                                                              ┆ ---     ┆ ---   │
│ str                                                                                                              ┆ i64     ┆ f64   │
╞══════════════════════════════════════════════════════════════════════════════════════════════════════════════════╪═════════╪═══════╡
│ ["DECISION_VISIBILITY_CONTENT_REMOVED"]                                                                          ┆ 8797210 ┆ 56.98 │
│ ["DECISION_VISIBILITY_OTHER"]                                                                                    ┆ 6418981 ┆ 41.57 │
│ No

**Finding.**  
- **TikTok `decision_visibility`:** 57% content removed (`CONTENT_REMOVED`), 42% `DECISION_VISIBILITY_OTHER` (primarily FYF demotion — "not eligible for For You feed recommendation"). Hard removal is the majority action.  
- **X `decision_visibility`:** 99.99% `DECISION_VISIBILITY_OTHER` — X almost never removes content outright; it labels it (NSFW, Bounce = soft demotion/restriction).  
- **Account-level actions:** TikTok's `decision_account` is 99.34% null — account actions are near-zero. X has 27.3% account suspensions and 0.12% terminations — account-level enforcement is central to X's approach.  
- **`decision_provision`:** TikTok uses `DECISION_PROVISION_PARTIAL_SUSPENSION` in 0.34% of cases; all other rows are null. Effectively unused on both platforms.

**Implication for thesis.**  
Platform modality (video vs. text) plausibly shapes enforcement type: TikTok's video-first architecture  
lends itself to content-level removal and FYF demotion; X's account-based model routes enforcement  
toward account suspension. This aligns with H1's affordance-theory framing and should be discussed in §4.

---
## 6  Territorial Scope — Geographic Granularity (→ RQ1, Data Methodology)

`territorial_scope` encodes which countries a decision applies to.  
This field also partly explains the volume asymmetry between platforms.

In [9]:
print('--- TikTok territorial_scope (top 5) ---')
print(topn(tt_topn, 'territorial_scope').select(['value', 'count', 'pct']).head(5))

print('\n--- X territorial_scope (top 5) ---')
print(topn(x_topn, 'territorial_scope').select(['value', 'count', 'pct']).head(5))

--- TikTok territorial_scope (top 5) ---
shape: (5, 3)
┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────┬───────┐
│ value                                                                                                                                                   ┆ count    ┆ pct   │
│ ---                                                                                                                                                     ┆ ---      ┆ ---   │
│ str                                                                                                                                                     ┆ i64      ┆ f64   │
╞═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╪══════════╪═══════╡
│ ["AT","BE","BG","CY","CZ","DE","DK","EE","ES","FI","FR","GR","HR","H

**Finding.**  
- **TikTok:** 98.83% of decisions apply to the same bloc of 29 EU/EEA countries simultaneously. Enforcement is pan-European by default; only ~1% are single-country.  
- **X:** All decisions are single-country. Top jurisdictions: DE (31.6%), FR (23.5%), ES (11.5%), NL (9.8%). X's DSA reporting is scoped to individual national jurisdictions.
- **X territorial scope is always exactly 6 characters** (`len_min = len_max = 6`), confirming single-country ISO codes like `["DE"]`.

**Implication for thesis.**  
This partly explains the volume gap: TikTok files one row per decision regardless of country coverage; X files one row per country per decision.  
The fields are not directly comparable — TikTok's `territorial_scope` contains JSON arrays; X's contains single ISO codes.  
For the Germany filter: a `contains("DE")` filter on TikTok captures pan-EU decisions affecting Germany alongside any DE-only rows.  
These are not Germany-specific enforcement actions — see thesis log Entry 001.  
Framing throughout the thesis must read: *"moderation decisions whose territorial scope includes Germany"*, not *"German moderation decisions"*.

---
## 7  Source Type — Who Triggered the Decision (→ RQ1)

`source_type` records whether the decision was proactively initiated by the platform or triggered by an external report.

In [10]:
print('--- TikTok source_type ---')
print(topn(tt_topn, 'source_type').select(['value', 'count', 'pct']))

print('\n--- X source_type ---')
print(topn(x_topn, 'source_type').select(['value', 'count', 'pct']))

--- TikTok source_type ---
shape: (4, 3)
┌────────────────────────────────┬──────────┬───────┐
│ value                          ┆ count    ┆ pct   │
│ ---                            ┆ ---      ┆ ---   │
│ str                            ┆ i64      ┆ f64   │
╞════════════════════════════════╪══════════╪═══════╡
│ SOURCE_VOLUNTARY               ┆ 15437671 ┆ 99.98 │
│ SOURCE_ARTICLE_16              ┆ 2179     ┆ 0.01  │
│ SOURCE_TYPE_OTHER_NOTIFICATION ┆ 136      ┆ 0.0   │
│ SOURCE_TRUSTED_FLAGGER         ┆ 4        ┆ 0.0   │
└────────────────────────────────┴──────────┴───────┘

--- X source_type ---
shape: (1, 3)
┌────────────────────────────────┬────────┬───────┐
│ value                          ┆ count  ┆ pct   │
│ ---                            ┆ ---    ┆ ---   │
│ str                            ┆ i64    ┆ f64   │
╞════════════════════════════════╪════════╪═══════╡
│ SOURCE_TYPE_OTHER_NOTIFICATION ┆ 670093 ┆ 100.0 │
└────────────────────────────────┴────────┴───────┘


**Finding.**  
- **TikTok:** 99.98% `SOURCE_VOLUNTARY` — own-initiative proactive enforcement. Only 0.01% Art. 16 user reports; a handful of Trusted Flagger reports.  
- **X:** 100% `SOURCE_TYPE_OTHER_NOTIFICATION` — `source_type` is a constant column for X. All decisions were triggered by an agent action (Bounce, NSFW, Suspend), not by user reports.
- `source_type` is a constant for X and therefore adds no variation for analysis.

**Data quality note on TikTok `decision_facts`.**  
TikTok's free-text `decision_facts` field contains a near-duplicate value pair:  
- `"The decision was taken pursuant to own-initiative investigations."` — 98.08% of rows  
- Same string with a trailing period removed — 1.23% of rows  

This is a minor normalization issue (trailing period variant) that should be cleaned before any text analysis.  
However, since `decision_facts` is not among the 9 thesis-relevant columns retained for analysis (CD-20260426-02), this has no impact on the thesis.

---
## 8  Content Type — Media Format (→ RQ1, Platform Modality)

`content_type` records the media format of the moderated content.  
This field is directly linked to the platform modality argument (video vs. text) at the heart of H1.

In [11]:
print('--- TikTok content_type ---')
print(topn(tt_topn, 'content_type').select(['value', 'count', 'pct']))

print('\n--- X content_type ---')
print(topn(x_topn, 'content_type').select(['value', 'count', 'pct']))

--- TikTok content_type ---
shape: (7, 3)
┌────────────────────────────────────────────┬─────────┬───────┐
│ value                                      ┆ count   ┆ pct   │
│ ---                                        ┆ ---     ┆ ---   │
│ str                                        ┆ i64     ┆ f64   │
╞════════════════════════════════════════════╪═════════╪═══════╡
│ ["CONTENT_TYPE_TEXT"]                      ┆ 7779495 ┆ 50.39 │
│ ["CONTENT_TYPE_VIDEO"]                     ┆ 6017717 ┆ 38.97 │
│ ["CONTENT_TYPE_IMAGE"]                     ┆ 1202425 ┆ 7.79  │
│ ["CONTENT_TYPE_OTHER"]                     ┆ 425475  ┆ 2.76  │
│ ["CONTENT_TYPE_AUDIO"]                     ┆ 14332   ┆ 0.09  │
│ ["CONTENT_TYPE_IMAGE","CONTENT_TYPE_TEXT"] ┆ 321     ┆ 0.0   │
│ ["CONTENT_TYPE_PRODUCT"]                   ┆ 225     ┆ 0.0   │
└────────────────────────────────────────────┴─────────┴───────┘

--- X content_type ---
shape: (3, 3)
┌──────────────────────────────────┬────────┬───────┐
│ value              

**Finding.**  
- **TikTok:** text (50%), video (39%), image (8%) — reflecting the platform's multi-format nature. Comments and captions are classified as text.  
- **X:** 99.97% `CONTENT_TYPE_SYNTHETIC_MEDIA` — an atypical classification. In X's DSA reporting, account-level actions affect an entire user profile rather than a specific post, and the platform encodes these as synthetic media. This is a platform-specific reporting convention, not a measure of AI-generated content prevalence.

**Implication for thesis.**  
X's `content_type` is analytically near-useless for comparing media formats — it is a constant encoding artefact.  
The platform modality comparison (RQ1) must therefore rely on platform identity itself as the modality proxy,  
not on `content_type` values. This limitation should be noted in §3.3 (operationalization) and §5.3.

---
## 9  Data Quality Issues Relevant to Analysis

This section records only quality issues that affect the 9 thesis-relevant columns  
or that must be disclosed in the methodology chapter.

### 9.1  Columns dropped — 100% null in both platforms

The following columns carry no analytical value and are excluded from the working dataset (see CD-20260426-02):

| Column | Status |
|--------|--------|
| `account_type` | 100% null (both) |
| `content_language` | 100% null (both) |
| `decision_ground_reference_url` | 100% null (both) |
| `decision_monetary` | 100% null (both) |
| `decision_monetary_other` | 100% null (both) |
| `end_date_account_restriction` | 100% null (both) |
| `end_date_monetary_restriction` | 100% null (both) |
| `incompatible_content_illegal` | 100% null (both) |
| `source_identity` | 100% null (both) |

**Interpretation:** Neither platform issued monetary restrictions in the profiling period.  
End-date fields being null everywhere indicates restrictions are applied without a stated end date (i.e., permanent or indefinite).  
This should be mentioned briefly in §3 but requires no further action.

### 9.2  Constant columns — no analytical variation

These columns have exactly 1 unique value across all rows and add nothing to analysis:

| Column | Platform | Constant value |
|--------|----------|----------------|
| `decision_ground` | X | `DECISION_GROUND_ILLEGAL_CONTENT` |
| `source_type` | X | `SOURCE_TYPE_OTHER_NOTIFICATION` |
| `automated_detection` | X | `No` |
| `platform_name` | TikTok | `"TikTok"` |

### 9.3  Normalizations required

| Issue | Action |
|-------|--------|
| X `platform_name`: `"X (formerly Twitter)"` in 0.22% of rows | Normalize to `"X"` before analysis |
| TikTok `decision_facts`: trailing period variant in 1.23% of rows | Not in thesis-relevant columns — no action needed |

### 9.4  Identifier integrity — confirmed clean

- **TikTok:** 15,439,990 unique UUIDs / 15,439,990 rows → no duplicates  
- **X:** 670,093 unique UUIDs / 670,093 rows → no duplicates  

Both datasets are free of duplicate rows at the identifier level.

### 9.5  Pre-2025 rows — backdated submissions

See thesis log Entry 002. A small fraction of rows have `application_date` outside 2025  
(TikTok: ~1.39M rows / 0.14%; X: ~3 rows / ~0%).  
These are platform backdated submissions — a documented DSA database characteristic.  
Rows outside the 2025 analysis window must be filtered by `application_date` before temporal analysis.

---
## 10  Scope Differences — Summary Table for Methodology Chapter

The following four structural differences are **not data quality issues** —  
they are genuine platform-level reporting differences that constrain the comparability of the two datasets.  
Each must be disclosed in §3 and revisited in §5.3 (Limitations).

| Dimension | TikTok | X | Implication |
|-----------|--------|---|-------------|
| **Decision ground** | 99.99% incompatible content (policy) | 100% illegal content (EU law) | Datasets measure different legal categories — not directly comparable |
| **Automation rate** | 96% fully automated decisions | 99% human decisions | Divergence reflects reporting scope, not just operational practice |
| **Source type** | 100% voluntary/proactive | 100% other notification | TikTok self-initiates; X reports externally triggered decisions |
| **Territorial scope** | Pan-EU bloc decisions (one row = 29 countries) | Single-country decisions (one row = one country) | Volume gap is partly a reporting unit artefact, not scale of enforcement |

---
## 11  Thesis-Relevant Columns — Final Selection

Based on the profiling results and research questions, the analytical dataset retains the following 9 columns  
(per cleaning decision CD-20260426-02). All other columns are dropped.

| Column | Role in analysis |
|--------|------------------|
| `uuid` | Unique record identifier — deduplication |
| `category` | Harm type — core variable for RQ1 and RQ2 |
| `content_type` | Media format — platform modality proxy (use with caveats from §8) |
| `automated_detection` | Automation flag — core variable for H1 |
| `automated_decision` | Automation flag — core variable for H1 |
| `territorial_scope` | Geographic filter — `contains("DE")` for Germany inclusion |
| `application_date` | Temporal dimension — event-window analysis for RQ2 |
| `platform_name` | Platform identifier — cross-platform comparison |
| `decision_visibility` | Enforcement action type — outcome variable for RQ1 |

**Note on `category` and schema version:**  
The DSA database introduced a schema break between API v1 and v2 in July 2025.  
Category field names changed between versions. The harmonized taxonomy mapping is documented in  
`taxonomy_mapping_v1_v2.csv` and `mapping_reference_table.md`. All longitudinal analysis must apply  
this mapping before aggregating across the full 2025 period.